# Assistant IA pour proches aidants\n\nPrototype éducatif avec données fictives pour organiser les tâches, estimer la fatigue et recommander des ressources.\n\n⚠️ Ne remplace pas un avis médical, social, psychologique ou juridique.

In [13]:
!pip install pandas scikit-learn -q
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [14]:
aidants = pd.DataFrame([
    {'prenom':'Maya','relation':'fille','heures_aide_semaine':18,'niveau_fatigue':7,'soutien_famille':'moyen'},
    {'prenom':'Louis','relation':'conjoint','heures_aide_semaine':35,'niveau_fatigue':9,'soutien_famille':'faible'},
    {'prenom':'Amina','relation':'soeur','heures_aide_semaine':10,'niveau_fatigue':4,'soutien_famille':'bon'}
])
aidants

,prenom,relation,heures_aide_semaine,niveau_fatigue,soutien_famille
0,Maya,fille,18,7,moyen
1,Louis,conjoint,35,9,faible
2,Amina,soeur,10,4,bon


In [16]:
def fatigue_risk(niveau_fatigue, heures_aide_semaine, soutien_famille):
    score = niveau_fatigue * 1.5 + heures_aide_semaine / 5
    if soutien_famille == 'faible':
        score += 3
    if score >= 18:
        return 'élevé'
    if score >= 11:
        return 'modéré'
    return 'faible'

aidants['risque_epuisement'] = aidants.apply(lambda r: fatigue_risk(r.niveau_fatigue, r.heures_aide_semaine, r.soutien_famille), axis=1)
aidants

,prenom,relation,heures_aide_semaine,niveau_fatigue,soutien_famille,risque_epuisement
0,Maya,fille,18,7,moyen,modéré
1,Louis,conjoint,35,9,faible,élevé
2,Amina,soeur,10,4,bon,faible


In [18]:
ressources = pd.DataFrame([
    {'nom':'Répit Solidaire','type':'répit','description':'Service fictif de répit à domicile.'},
    {'nom':'Ligne Aidants Écoute','type':'écoute','description':'Écoute confidentielle et orientation.'},
    {'nom':'Transport Santé Communautaire','type':'transport','description':'Transport pour rendez-vous médicaux.'},
])

def recommander(besoin, df):
    textes = df['type'] + ' ' + df['description']
    vectorizer = TfidfVectorizer()
    matrix = vectorizer.fit_transform(textes)
    scores = cosine_similarity(vectorizer.transform([besoin]), matrix).flatten()
    result = df.copy()
    result['score'] = scores
    return result.sort_values('score', ascending=False)

recommander('besoin de répit et écoute', ressources)

,nom,type,description,score
1,Ligne Aidants Écoute,écoute,Écoute confidentielle et orientation.,0.566947
0,Répit Solidaire,répit,Service fictif de répit à domicile.,0.530330
2,Transport Santé Communautaire,transport,Transport pour rendez-vous médicaux.,0.000000
